# Parse cached Pfam GFFs → Parquet (issue #88)

Run **after** `scripts/download_to_cache.py` has populated `loaded_pfam_gff/raw_cache/`.

Reads `loaded_pfam_gff/manifest.csv` and the cached GFF files, parses
the 9-column NMDC-flavor GFF in **streaming batches**, and writes one
Parquet:

- `loaded_pfam_gff/pfam_annotation_gff.parquet`

**Why streaming.** ~3 billion total rows across ~4,800 files. We use
`pyarrow.parquet.ParquetWriter` with a fixed schema, parse in batches of
~500 MB raw text per RowGroup, and drop after each flush. Peak memory
stays bounded.

**Schema choices** (issue #88):
- `pfam_accession` is the version-less Pfam ID (e.g. `PF02992`) — joins
  directly to `nmdc_ref_data.pfam_terms.pfam_id`.
- `pfam_name` is **not stored** — redundant with `pfam_terms.name` and saves
  ~60 GB across 3 B rows.
- Cols 2 (`source`, always "HMMER 3.1b2 (February 2015)"), 7-8 (strand/phase
  placeholders, always "."), and the `ID=` / `fake_percent_id=` attributes
  are dropped — constants or non-data.

**No network I/O.** Cache misses are reported at the end — no HTTP retries
from the kernel.

### Imports + config + logger

In [ ]:
import logging
import os
import re
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

BATCH_TARGET_BYTES = int(os.environ.get("PARSE_BATCH_TARGET_BYTES", str(500 * 1024 * 1024)))

OUT_DIR = Path(os.environ.get("PFAM_GFF_OUT_DIR", "loaded_pfam_gff"))
if not OUT_DIR.exists():
    raise RuntimeError(f"OUT_DIR does not exist: {OUT_DIR.resolve()}")

CACHE_DIR = Path(os.environ.get("DOWNLOAD_CACHE_DIR", str(OUT_DIR / "raw_cache")))
if not CACHE_DIR.exists():
    raise RuntimeError(
        f"CACHE_DIR does not exist: {CACHE_DIR.resolve()}. "
        "Run scripts/download_to_cache.py first."
    )

LOG_DIR  = OUT_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)
LOG_FILE = LOG_DIR / f"parse_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logger = logging.getLogger("nmdc_lakehouse.pfam_gff.parse")
logger.setLevel(logging.INFO)
for _h in list(logger.handlers):
    logger.removeHandler(_h)
_fh = logging.FileHandler(LOG_FILE)
_fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
logger.addHandler(_fh)
_sh = logging.StreamHandler()
_sh.setFormatter(logging.Formatter("%(message)s"))
logger.addHandler(_sh)
logger.propagate = False

print(f"OUT_DIR:   {OUT_DIR.resolve()}")
print(f"CACHE_DIR: {CACHE_DIR.resolve()}")
print(f"LOG_FILE:  {LOG_FILE.resolve()}")
print(f"BATCH_TARGET_BYTES: {BATCH_TARGET_BYTES:,}")

### Load the manifest

In [ ]:
manifest_csv = OUT_DIR / "manifest.csv"
if not manifest_csv.exists():
    raise RuntimeError(
        f"manifest.csv not found at {manifest_csv.resolve()}. "
        "Run fetch_pfam_gff.ipynb first."
    )
manifest = pd.read_csv(manifest_csv)
logger.info(f"manifest loaded: {len(manifest):,} rows from {manifest_csv}")

### Pfam GFF parser (9 columns, no header)

Format per line (tab-separated):
```
gene_id  source  pfam_accession  start  end  score  .  .  ID=...;Name=...;e-value=...;alignment_length=...;model_start=...;model_end=...
```

We keep: `gene_id`, `pfam_accession`, `start`, `end`, `score`, plus four
values pulled from the col-9 attribute string: `e_value`, `alignment_length`,
`model_start`, `model_end`.

In [ ]:
_GFF_COLS = [
    "gene_id", "_source", "pfam_accession",
    "start", "end", "score",
    "_strand", "_phase", "_attrs",
]

# Match e.g. 'e-value=1.9e-14' or 'alignment_length=140' inside the attrs string.
_ATTR_RE = re.compile(r"(?:^|;)([A-Za-z_][A-Za-z0-9_\-]*)=([^;]*)")

_WANTED_ATTRS = {
    "e-value":          ("e_value",          float),
    "alignment_length": ("alignment_length", "Int64"),
    "model_start":      ("model_start",      "Int64"),
    "model_end":        ("model_end",        "Int64"),
}


def _extract_attrs(series: pd.Series) -> pd.DataFrame:
    """Pull the four wanted key=value pairs out of the attribute strings."""
    extracted = {col: [None] * len(series) for _, (col, _) in _WANTED_ATTRS.items()}
    for i, attrs in enumerate(series):
        if not isinstance(attrs, str):
            continue
        for m in _ATTR_RE.finditer(attrs):
            spec = _WANTED_ATTRS.get(m.group(1))
            if spec is not None:
                extracted[spec[0]][i] = m.group(2)
    return pd.DataFrame(extracted)


def _parse_pfam_gff(text: str) -> pd.DataFrame | None:
    if not text or not text.strip():
        return None
    import io
    df = pd.read_csv(
        io.StringIO(text),
        sep="\t",
        header=None,
        names=_GFF_COLS,
        comment="#",  # skip any GFF3 ## directives if present
        on_bad_lines="warn",
        engine="c",
        dtype=str,
    )
    if df.empty:
        return None
    attrs = _extract_attrs(df["_attrs"])
    out = pd.DataFrame({
        "gene_id":          df["gene_id"],
        "pfam_accession":   df["pfam_accession"],
        "start":            df["start"],
        "end":              df["end"],
        "score":            df["score"],
        "e_value":          attrs["e_value"],
        "alignment_length": attrs["alignment_length"],
        "model_start":      attrs["model_start"],
        "model_end":        attrs["model_end"],
    })
    return out

### Cache reader + streaming parse loop

In [ ]:
def _cache_path_for(url: str) -> Path:
    return CACHE_DIR / urlparse(url).path.lstrip("/")


def _read_and_parse(row):
    path = _cache_path_for(row.url)
    if not path.exists():
        return None, (row.url, "cache miss")
    try:
        text = path.read_text()
        df = _parse_pfam_gff(text)
        if df is None:
            return None, None  # empty file — not an error
        df["workflow_run_id"] = row.was_generated_by
        df["data_object_id"]  = row.id
        return df, None
    except Exception as exc:
        return None, (row.url, str(exc))


ARROW_SCHEMA = pa.schema([
    ("gene_id",          pa.string()),
    ("pfam_accession",   pa.string()),
    ("start",            pa.int64()),
    ("end",              pa.int64()),
    ("score",            pa.float64()),
    ("e_value",          pa.float64()),
    ("alignment_length", pa.int64()),
    ("model_start",      pa.int64()),
    ("model_end",        pa.int64()),
    ("workflow_run_id",  pa.string()),
    ("data_object_id",   pa.string()),
])

_INT_COLS    = ["start", "end", "alignment_length", "model_start", "model_end"]
_FLOAT_COLS  = ["score", "e_value"]
_STRING_COLS = ["gene_id", "pfam_accession", "workflow_run_id", "data_object_id"]


def _coerce_for_arrow(df: pd.DataFrame) -> pd.DataFrame:
    for col in _INT_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    for col in _FLOAT_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in _STRING_COLS:
        df[col] = df[col].astype("string")
    return df


def _flush(writer: pq.ParquetWriter, frames: list[pd.DataFrame]) -> int:
    if not frames:
        return 0
    df = pd.concat(frames, ignore_index=True)
    df = _coerce_for_arrow(df)
    table = pa.Table.from_pandas(df, schema=ARROW_SCHEMA, preserve_index=False)
    writer.write_table(table)
    return len(df)


def stream_to_parquet(rows: pd.DataFrame, out_path: Path) -> dict:
    """Stream-parse all cached GFFs into one Parquet.

    At most BATCH_TARGET_BYTES of raw text is held in DataFrames at any time;
    each batch is concatenated, coerced to ARROW_SCHEMA, written as a Parquet
    RowGroup, and dropped.
    """
    stats = {"rows": 0, "files_with_data": 0, "empties": 0,
             "cache_misses": 0, "parse_errors": 0, "batches": 0}
    cache_misses_sample: list[str] = []
    parse_errors_sample: list[tuple[str, str]] = []

    total = len(rows)
    cache_hits = sum(1 for r in rows.itertuples() if _cache_path_for(r.url).exists())
    logger.info(f"  cache: {cache_hits:,}/{total:,} present; {total - cache_hits:,} missing")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        out_path.unlink()  # ParquetWriter cannot append to an existing file

    batch_frames: list[pd.DataFrame] = []
    batch_bytes = 0

    with pq.ParquetWriter(out_path, ARROW_SCHEMA, compression="snappy") as writer:
        for i, row in enumerate(rows.itertuples(), 1):
            df, err = _read_and_parse(row)
            if df is not None:
                batch_frames.append(df)
                stats["files_with_data"] += 1
                cache_path = _cache_path_for(row.url)
                if cache_path.exists():
                    batch_bytes += cache_path.stat().st_size
            elif err is not None:
                if err[1] == "cache miss":
                    stats["cache_misses"] += 1
                    if len(cache_misses_sample) < 10:
                        cache_misses_sample.append(err[0])
                else:
                    stats["parse_errors"] += 1
                    if len(parse_errors_sample) < 10:
                        parse_errors_sample.append(err)
            else:
                stats["empties"] += 1

            if batch_bytes >= BATCH_TARGET_BYTES:
                n = _flush(writer, batch_frames)
                stats["rows"] += n
                stats["batches"] += 1
                logger.info(
                    f"    batch {stats['batches']}: {n:,} rows "
                    f"(file {i}/{total}, total rows so far {stats['rows']:,})"
                )
                batch_frames = []
                batch_bytes = 0

            if i % 200 == 0 or i == total:
                print(f"  {i}/{total}…", end="", flush=True)

        # final flush
        n = _flush(writer, batch_frames)
        if n:
            stats["rows"] += n
            stats["batches"] += 1
            logger.info(f"    batch {stats['batches']} (final): {n:,} rows")
    print()

    logger.info(
        f"  parsed: {stats['files_with_data']:,} files with data → "
        f"{stats['rows']:,} rows in {stats['batches']:,} row groups; "
        f"{stats['empties']:,} empties; "
        f"{stats['cache_misses']:,} cache-misses; "
        f"{stats['parse_errors']:,} parse-errors"
    )
    if cache_misses_sample:
        for url in cache_misses_sample[:5]:
            logger.info(f"    cache-miss: {url}")
        if stats["cache_misses"] > 5:
            logger.info(f"    … and {stats['cache_misses'] - 5} more cache-misses")
    if parse_errors_sample:
        for url, msg in parse_errors_sample[:5]:
            logger.info(f"    parse-error: {url}: {msg}")
        if stats["parse_errors"] > 5:
            logger.info(f"    … and {stats['parse_errors'] - 5} more parse-errors")

    return stats

### Run

In [ ]:
out_path = OUT_DIR / "pfam_annotation_gff.parquet"
logger.info(f"\n{'─'*60}")
logger.info(f"Pfam Annotation GFF ({len(manifest):,} files → {out_path.name})")

t_run = time.monotonic()
stats = stream_to_parquet(manifest, out_path)
elapsed = time.monotonic() - t_run

if stats["rows"] == 0:
    logger.info("  no rows — removing empty Parquet")
    if out_path.exists():
        out_path.unlink()
else:
    size_gb = out_path.stat().st_size / 1024**3
    logger.info(f"  wrote {out_path} ({size_gb:,.2f} GB)")

logger.info(f"\n{'='*60}")
logger.info(f"Done in {elapsed/60:.1f} min")
logger.info(
    f"  {stats['rows']:,} rows from {stats['files_with_data']:,} files "
    f"({stats['batches']:,} row groups)"
)
logger.info(f"Full log: {LOG_FILE}")

### Spot-check

Reads only the first row group — `pd.read_parquet(p)` would load the whole
thing and OOM the pod at this scale.

In [ ]:
p = OUT_DIR / "pfam_annotation_gff.parquet"
if not p.exists():
    print("-- pfam_annotation_gff.parquet: not written")
else:
    pf     = pq.ParquetFile(p)
    md     = pf.metadata
    nrows  = md.num_rows
    nrg    = md.num_row_groups
    sample = pf.read_row_group(0).to_pandas()
    print(f"=== pfam_annotation_gff.parquet: {nrows:,} rows / {nrg} row group(s) ===")
    print(sample.dtypes)
    print(sample.head(3).to_string())
    print(f"first-row-group distinct workflow runs: {sample['workflow_run_id'].nunique():,}")
    print(f"first-row-group distinct pfam accessions: {sample['pfam_accession'].nunique():,}")

## Next steps

Run **`ingest_pfam_gff.ipynb`** to upload to MinIO bronze
(`s3a://cdm-lake/tenant-general-warehouse/nmdc/datasets/results/`) and register
`nmdc_results.pfam_annotation_gff` as a Delta table.